In [1]:
import os
# Change working root directory
os.chdir("/home/antoine/Bureau/detectives/detectives_minimal_implementation")

In [2]:
propp_outputs_directory = "data/texts"

In [3]:
# ! pip install booknlp_fr -U
from booknlp_fr import (
    load_tokenizer_and_embedding_model,
    get_embedding_tensor_from_tokens_df,
    load_text_file,
    load_tokens_df,
    save_tokens_df,
    load_entities_df,
)
from tqdm.auto import tqdm
import torch
import os

/home/antoine/Bureau/character_attributes_classification/.venv/lib/python3.10/site-packages/booknlp_fr/booknlp_fr_add_entities_features.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


booknlp_fr package loaded successfully.


In [4]:
def extract_char_atts(row):
    result = []
    if row['char_att_agent'] != -1:
        result.append('agent')
    if row['char_att_patient'] != -1:
        result.append('patient')
    if row['char_att_mod'] != -1:
        result.append('mod')
    if row['char_att_poss'] != -1:
        result.append('poss')
    return result

In [5]:
tokenizer, embedding_model = load_tokenizer_and_embedding_model(model_name='almanach/camembert-large')

Some weights of CamembertModel were not initialized from the model checkpoint at almanach/camembert-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizer and Embedding Model Initialized: almanach/camembert-large


In [6]:
embeddings_directory = propp_outputs_directory.replace("texts", "attribute_embeddings")
if not os.path.exists(embeddings_directory):
    os.makedirs(embeddings_directory)

extension = ".book"
book_files = sorted([f.replace(extension, "") for f in os.listdir(propp_outputs_directory) if f.endswith(extension)], reverse=False)

for file_name in tqdm(book_files[:]):
    attribute_embeddings_tensor_path = os.path.join(embeddings_directory, f"{file_name}.attribute_embeddings")

    if not os.path.exists(attribute_embeddings_tensor_path):

        txt_content = load_text_file(file_name, propp_outputs_directory)
        tokens_df = load_tokens_df(file_name, propp_outputs_directory)
        entities_df = load_entities_df(file_name, propp_outputs_directory)

        tokens_embedding_tensor = get_embedding_tensor_from_tokens_df(txt_content,
                                                                      tokens_df,
                                                                      tokenizer,
                                                                      embedding_model,
                                                                      sliding_window_size='max',
                                                                      mini_batch_size= 16,
                                                                      sliding_window_overlap=0.5,
                                                                      subword_pooling_strategy="first_last")

        att_cols = ["char_att_agent", "char_att_patient", "char_att_mod", "char_att_poss"]
        attributes_tokens_df = tokens_df[tokens_df[att_cols].ne(-1).any(axis=1)].copy()
        attributes_tokens_df['booknlp_categories'] = attributes_tokens_df.apply(extract_char_atts, axis=1)

        PER_entities_df = entities_df[entities_df["cat"] == "PER"]
        PER_entities_head_ids = PER_entities_df["head_id"].tolist()
        attributes_tokens_df = attributes_tokens_df[(attributes_tokens_df["char_att_agent"].isin(PER_entities_head_ids))
                                                     | (attributes_tokens_df["char_att_patient"].isin(PER_entities_head_ids))
                                                     | (attributes_tokens_df["char_att_mod"].isin(PER_entities_head_ids))
                                                     | (attributes_tokens_df["char_att_poss"].isin(PER_entities_head_ids))]

        attributes_indexes = attributes_tokens_df.index.tolist()

        tokens_df["is_PER_attribute"] = 0
        tokens_df.loc[attributes_indexes, "is_PER_attribute"] = 1
        save_tokens_df(tokens_df,file_name,propp_outputs_directory)

        attribute_embeddings = tokens_embedding_tensor[attributes_indexes]
        torch.save(attribute_embeddings, attribute_embeddings_tensor_path)

  0%|          | 0/379 [00:00<?, ?it/s]